# Notebook 10 — Multivariate DLMs and Missing Data

**References:** W&H §16.1–16.3; Petris §2.7, §4.3

**New engine functions:**
- `engine.filter.kalman_filter_missing(spec, y)` — handles NaN observations
- `engine.models.make_multivariate_local_level(p, V, W_level)` — p series, one shared level

**Two parts:**
- **Part A:** Missing-data filter — imputation and uncertainty quantification
- **Part B:** Multivariate local level — shared scalar level across multiple series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from engine.filter import kalman_filter, kalman_filter_missing
from engine.models import make_local_level, make_multivariate_local_level
from engine.simulate import simulate

## Part A: Missing-Data Filter

### Theory

When $y_t$ contains NaN entries, the update step is skipped:

$$m_t = a_t, \quad C_t = R_t \quad \text{(posterior = prior)}$$

No log-likelihood contribution is added. The filter continues forward —
uncertainty grows at rate $W$ per missing step.

This gives exact Bayesian imputation: the predictive distribution over
a missing gap follows the state evolution equations only.

**Reference:** W&H §16.1; Petris §2.7

In [ ]:
spec = make_local_level(V=1.0, W_level=0.2)
sim = simulate(spec, n=100, seed=0)
y_miss = sim.y.copy()
gap_start, gap_end = 40, 60
y_miss[gap_start:gap_end] = np.nan

fr_miss = kalman_filter_missing(spec, y_miss)

t = np.arange(100)
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(t,
    fr_miss.f[:, 0] - 1.96 * np.sqrt(fr_miss.Q[:, 0, 0]),
    fr_miss.f[:, 0] + 1.96 * np.sqrt(fr_miss.Q[:, 0, 0]),
    alpha=0.25, color="steelblue", label="95% predictive CI")
ax.plot(t, sim.y[:, 0], "k.", ms=3, label="true obs")
ax.plot(t[np.isnan(y_miss[:, 0])], sim.y[np.isnan(y_miss[:, 0]), 0],
        "r.", ms=4, label="hidden obs (gap)")
ax.plot(t, fr_miss.f[:, 0], "b-", lw=1.0, label="one-step forecast (imputation)")
ax.axvspan(gap_start, gap_end - 1, alpha=0.08, color="red", label="missing gap")
ax.legend(fontsize=8)
ax.set_title("Missing-data filter: imputation over 20-step gap")
plt.tight_layout()
plt.show()
print(f"CI width at gap midpoint (t=50): {2 * 1.96 * np.sqrt(fr_miss.Q[50, 0, 0]):.3f}")

## Part B: Multivariate Local Level

### Model

$p$ series share a common scalar level $\theta_t$:

$$y_{i,t} = \theta_t + v_{i,t}, \quad v_{i,t} \sim N(0, \sigma^2_i), \quad i=1,\ldots,p$$
$$\theta_t = \theta_{t-1} + w_t, \quad w_t \sim N(0, W_\text{level})$$

In matrix form: $F = \mathbf{1}_p$, $G = [1]$, state dim $d = 1$ regardless of $p$.

**When is this appropriate?** When the series share a true common component.
Compare to $p$ independent local-level models using log marginal likelihoods.

**Reference:** W&H §16.2; Petris §4.3

In [ ]:
rng = np.random.default_rng(42)
T = 100
level = np.cumsum(rng.normal(scale=np.sqrt(0.3), size=T))
y3 = np.column_stack([
    level + rng.normal(scale=1.0, size=T),
    level + rng.normal(scale=2.0, size=T),
    level + rng.normal(scale=0.5, size=T),
])

spec_shared = make_multivariate_local_level(p=3, V=[1.0, 4.0, 0.25], W_level=0.3)
fr_shared = kalman_filter(spec_shared, y3)
print(f"Shared-level log-lik: {fr_shared.loglik:.2f}")
print(f"Filtered state shape: {fr_shared.m.shape}  (T=100, d=1)")

In [ ]:
spec_ind1 = make_local_level(V=1.0, W_level=0.3)
spec_ind2 = make_local_level(V=4.0, W_level=0.3)
spec_ind3 = make_local_level(V=0.25, W_level=0.3)
fr1 = kalman_filter(spec_ind1, y3[:, [0]])
fr2 = kalman_filter(spec_ind2, y3[:, [1]])
fr3 = kalman_filter(spec_ind3, y3[:, [2]])
loglik_ind = fr1.loglik + fr2.loglik + fr3.loglik
print(f"Independent models log-lik sum: {loglik_ind:.2f}")
print(f"Shared-level log-lik          : {fr_shared.loglik:.2f}")
print(f"Log Bayes factor (shared vs ind): {fr_shared.loglik - loglik_ind:.2f}")

In [ ]:
t_arr = np.arange(T)
fig, ax = plt.subplots(figsize=(11, 4))
for i, col in enumerate(["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(t_arr, y3[:, i], ".", ms=2, color=col, alpha=0.5, label=f"y_{i+1}")
ax.plot(t_arr, level, "k-", lw=1.5, label="true shared level")
ax.plot(t_arr, fr_shared.m[:, 0], "r--", lw=1.5, label="filtered shared level")
ax.legend(fontsize=8)
ax.set_title("Multivariate local level: shared scalar level")
plt.tight_layout()
plt.show()

## Exercises

**Exercise 1** — Bivariate imputation:
Simulate a bivariate series (p=2) with `make_multivariate_local_level(p=2, V=[1.0, 2.0], W_level=0.3)`.
Set 20% of observations in series 1 to NaN. Run `kalman_filter_missing` and
plot the imputed values with 95% credible intervals.

```python
# YOUR CODE HERE
# spec2 = make_multivariate_local_level(p=2, V=[1.0, 2.0], W_level=0.3)
# sim2 = ...
# Set 20% of y[:, 0] to NaN
# fr_miss2 = kalman_filter_missing(spec2, y_miss2)
```

**Exercise 2** — Two independent levels:
Compare the log marginal likelihood of the shared-level model to two independent
`make_local_level` models run separately. Generate data where they share a true
common level — show the Bayes factor favours the shared model.

**Exercise 3 (challenge)** — Dynamic factor model concept:
For p=4 series and k=2 latent factors, write down the shapes of F, G, V, W
if each factor is an independent random walk. What is the state dimension d?

In [ ]:
# Exercise 1 scaffold
# YOUR CODE HERE